In [6]:
import numpy as np
import matplotlib.pyplot as plt
import os
from tqdm.auto import tqdm


path = os.path.join("..","rpg_vid2e","data","upsample_input_smaller_events","seq0")
events = sorted([os.path.join(path, file) for file in os.listdir(path)])

os.listdir(path)

['0000000938.npz',
 '0000000203.npz',
 '0000000208.npz',
 '0000002810.npz',
 '0000000149.npz',
 '0000002702.npz',
 '0000000827.npz',
 '0000002720.npz',
 '0000002910.npz',
 '0000000830.npz',
 '0000001727.npz',
 '0000003712.npz',
 '0000000080.npz',
 '0000003824.npz',
 '0000000409.npz',
 '0000000257.npz',
 '0000002371.npz',
 '0000000835.npz',
 '0000000253.npz',
 '0000002690.npz',
 '0000000145.npz',
 '0000000311.npz',
 '0000001024.npz',
 '0000000414.npz',
 '0000001321.npz',
 '0000003316.npz',
 '0000002618.npz',
 '0000000357.npz',
 '0000001225.npz',
 '0000001439.npz',
 '0000000142.npz',
 '0000003652.npz',
 '0000000298.npz',
 '0000002025.npz',
 '0000000199.npz',
 '0000000824.npz',
 '0000002904.npz',
 '0000001227.npz',
 '0000002650.npz',
 '0000001157.npz',
 '0000000470.npz',
 '0000002911.npz',
 '0000002300.npz',
 '0000001199.npz',
 '0000003693.npz',
 '0000003278.npz',
 '0000003207.npz',
 '0000000733.npz',
 '0000002991.npz',
 '0000002542.npz',
 '0000002238.npz',
 '0000000678.npz',
 '0000003547

In [2]:
print(os.listdir(path = os.path.join("..","rpg_vid2e","data","upsample_input_smaller_events", "seq0")))

['0000000938.npz', '0000000203.npz', '0000000208.npz', '0000002810.npz', '0000000149.npz', '0000002702.npz', '0000000827.npz', '0000002720.npz', '0000002910.npz', '0000000830.npz', '0000001727.npz', '0000003712.npz', '0000000080.npz', '0000003824.npz', '0000000409.npz', '0000000257.npz', '0000002371.npz', '0000000835.npz', '0000000253.npz', '0000002690.npz', '0000000145.npz', '0000000311.npz', '0000001024.npz', '0000000414.npz', '0000001321.npz', '0000003316.npz', '0000002618.npz', '0000000357.npz', '0000001225.npz', '0000001439.npz', '0000000142.npz', '0000003652.npz', '0000000298.npz', '0000002025.npz', '0000000199.npz', '0000000824.npz', '0000002904.npz', '0000001227.npz', '0000002650.npz', '0000001157.npz', '0000000470.npz', '0000002911.npz', '0000002300.npz', '0000001199.npz', '0000003693.npz', '0000003278.npz', '0000003207.npz', '0000000733.npz', '0000002991.npz', '0000002542.npz', '0000002238.npz', '0000000678.npz', '0000003547.npz', '0000002423.npz', '0000000742.npz', '00000019

In [7]:
import cv2
import numpy as np
import matplotlib.cm as cm

frame_width = 320
frame_height = 240
fps = 25
video_name = "events.mp4"
j = 0



In [14]:
np.load(events[0])["p"]

array([1, 1])

In [ ]:
def process_video(data, bin_time_s=-1, frame_width=320, frame_height=240, fps=25, video_name="events.mp4"):
    l = len(data)

    # Define codec and create VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # use 'mp4v' for .mp4 files
    out = cv2.VideoWriter(video_name, fourcc, fps, (frame_width, frame_height))

    curr_time = 0
    if bin_time_s == -1:
        bin_ns = float("inf")
    else:
        bin_ns = bin_time_s * 1e9

    frames = []
    j = 0
    ts_all = []
    curr_frame = np.zeros((frame_height, frame_width, 1))
    curr_t = 0
    cnt = 0

    pb = tqdm(data)
    print(bin_ns)
    for frame in pb:

        data = np.load(frame)
        xs,ys,ts, ps = data['x'], data['y'], data['t'], data['p']

        for i in range(len(xs)):
            x, y, t, p = xs[i], ys[i], ts[i], ps[i]

            if t > curr_t + bin_ns:
                curr_t += bin_ns
                cnt += 1
                pb.set_description("Processing frame {}, no of frames saved = {}".format(i, cnt))
                empty_mask = (curr_frame == 0).squeeze(2)

                # Normalize to -1..1 range for better bwr mapping
                norm = cv2.normalize(curr_frame, None, -1, 1, cv2.NORM_MINMAX)

                # Apply matplotlib's 'bwr' colormap
                colormapped = cm.bwr((norm.squeeze() + 1) / 2)  # scale to [0, 1]
                colormapped = (colormapped[:, :, :3] * 255).astype(np.uint8)  # drop alpha, scale to 0-255

                colormapped[empty_mask] = 0

                # Convert RGB (matplotlib) to BGR (OpenCV)
                colormapped_bgr = cv2.cvtColor(colormapped, cv2.COLOR_RGB2BGR)

                out.write(colormapped_bgr)

                # zero frame
                curr_frame = np.zeros((frame_height, frame_width, 1))

            if p == 1:
                curr_frame[y][x][0] += 1
            else:
                curr_frame[y][x][0] -= 1
        j += 1

    empty_mask = (curr_frame == 0).squeeze(2)
    # Normalize to -1..1 range for better bwr mapping
    norm = cv2.normalize(curr_frame, None, -1, 1, cv2.NORM_MINMAX)

    # Apply matplotlib's 'bwr' colormap
    colormapped = cm.bwr((norm.squeeze() + 1) / 2)  # scale to [0, 1]
    colormapped = (colormapped[:, :, :3] * 255).astype(np.uint8)  # drop alpha, scale to 0-255

    colormapped[empty_mask] = 0

    # Convert RGB (matplotlib) to BGR (OpenCV)
    colormapped_bgr = cv2.cvtColor(colormapped, cv2.COLOR_RGB2BGR)

    out.write(colormapped_bgr)

    out.release()
    return ts_all


ts = process_video(events, bin_time_s=0.025, video_name="events_2.mp4")

In [5]:
def process_video_no_time(data, frame_width=320, frame_height=240, fps=25, video_name="events_no_time.mp4"):
    l = len(data)

    # Define codec and create VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # use 'mp4v' for .mp4 files
    out = cv2.VideoWriter(video_name, fourcc, fps, (frame_width, frame_height))


    pb = tqdm(data)
    for frame in pb:

        curr_frame = np.zeros((frame_height, frame_width, 1))
        data = np.load(frame)
        xs, ys, ts, ps = data['x'], data['y'], data['t'], data['p']

        for i in range(len(xs)):
            x, y, t, p = xs[i], ys[i], ts[i], ps[i]

            if p == 1:
                curr_frame[y][x][0] += 1
            else:
                curr_frame[y][x][0] -= 1

        empty_mask = (curr_frame == 0).squeeze(2)

        # Normalize to -1..1 range for better bwr mapping
        norm = cv2.normalize(curr_frame, None, -1, 1, cv2.NORM_MINMAX)

        # Apply matplotlib's 'bwr' colormap
        colormapped = cm.bwr((norm.squeeze() + 1) / 2)  # scale to [0, 1]
        colormapped = (colormapped[:, :, :3] * 255).astype(np.uint8)  # drop alpha, scale to 0-255

        colormapped[empty_mask] = 0

        # Convert RGB (matplotlib) to BGR (OpenCV)
        colormapped_bgr = cv2.cvtColor(colormapped, cv2.COLOR_RGB2BGR)

        out.write(colormapped_bgr)

    out.release()


ts = process_video_no_time(events, video_name="events.mp4")


  0%|          | 0/3867 [00:00<?, ?it/s]